# LangGraph Code Assistant - Project Demonstration

A self-correcting AI code assistant built with LangGraph that:
- Generates structured Python code using LLMs (OpenAI, Mistral, local models)
- Validates code in an isolated subprocess with timeout protection
- Automatically retries and corrects failed code generation
- Optionally retrieves project-aware context using Qdrant RAG index
- Exposes both CLI and web API interfaces with FastAPI

**Key Features:**
- Multi-provider support (OpenAI, Mistral, OpenRouter, local HuggingFace)
- Configurable retry logic with up to 3 correction attempts
- Project-aware RAG retrieval for better context-aware solutions
- FastAPI backend with authentication and rate limiting
- Benchmarking and performance tracking
- Optional sandboxed code execution for safer validation

## 1. Installation and Setup

In [ ]:
# Install dependencies
# Uncomment and run to install required packages
# !pip install -r requirements.txt

# Alternatively, install specific packages:
# %pip install langchain langgraph langchain-mistralai langchain-openai langchain-huggingface faiss-cpu beautifulsoup4 fastapi uvicorn python-dotenv pydantic qdrant-client

print("✓ Dependencies installed successfully")

## 2. Environment Configuration

In [ ]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# Load environment variables from .env
load_dotenv()

# Configuration options for the Code Assistant
class CodeAssistantConfig:
    """Configuration for the Code Assistant."""
    
    # LLM Provider settings
    PROVIDER = os.getenv("PROVIDER", "mistral")  # Options: mistral, openai, local, openrouter
    MODEL_NAME = os.getenv("MODEL_NAME", "mistral-medium-latest")
    TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))
    
    # API Keys (from .env)
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "")
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
    
    # Execution settings
    MAX_ITERATIONS = int(os.getenv("MAX_ITERATIONS", "3"))
    VALIDATION_TIMEOUT = int(os.getenv("VALIDATION_TIMEOUT", "5"))
    
    # RAG settings
    RAG_ENABLED = os.getenv("RAG_ENABLED", "false").lower() == "true"
    RAG_COLLECTION = os.getenv("RAG_COLLECTION", "code-assistant-project")
    RAG_K = int(os.getenv("RAG_K", "4"))

# Define structured output schema
class CodeSolution(BaseModel):
    """Schema for code solutions."""
    prefix: str = Field(description="Description of the problem and approach")
    imports: str = Field(description="Code block import statements")
    code: str = Field(description="The functional code block")

print("✓ Configuration loaded successfully")
print(f"  Provider: {CodeAssistantConfig.PROVIDER}")
print(f"  Model: {CodeAssistantConfig.MODEL_NAME}")
print(f"  Max Iterations: {CodeAssistantConfig.MAX_ITERATIONS}")
print(f"  RAG Enabled: {CodeAssistantConfig.RAG_ENABLED}")

## 3. RAG Components - Optional Project Context Retrieval

In [ ]:
from pathlib import Path

# RAG Setup - Load project-aware context from Qdrant vector database
def setup_rag_retriever(collection_name: str = "code-assistant-project", qdrant_path: str = "data/qdrant"):
    """
    Initialize the RAG retriever if Qdrant index exists.
    This retrieves relevant project context for better code generation.
    """
    try:
        from qdrant_client import QdrantClient
        from langchain_mistralai.embeddings import MistralAIEmbeddings
        from langchain_community.vectorstores import Qdrant
        
        # Check if Qdrant collection exists
        qdrant_db_path = Path(qdrant_path)
        if not qdrant_db_path.exists():
            print(f"⚠ Qdrant database not found at {qdrant_path}")
            print("  To create it, run: python scripts/index_project_rag.py")
            return None
        
        # Initialize Qdrant client
        client = QdrantClient(path=qdrant_path)
        embeddings = MistralAIEmbeddings(model="mistral-embed")
        
        # Create retriever
        vectorstore = Qdrant(
            client=client,
            collection_name=collection_name,
            embeddings=embeddings,
        )
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        print(f"✓ RAG retriever initialized with collection: {collection_name}")
        return retriever
    except Exception as e:
        print(f"⚠ Could not initialize RAG: {e}")
        return None

# Initialize RAG (optional)
rag_retriever = None
if CodeAssistantConfig.RAG_ENABLED:
    rag_retriever = setup_rag_retriever()
else:
    print("ℹ RAG is disabled. Enable by setting RAG_ENABLED=true in .env")

## 4. Graph State and Node Functions

In [ ]:
import io
import contextlib
import subprocess
import tempfile
from typing import Annotated, TypedDict, Literal, Any
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from langchain_mistralai import ChatMistralAI

# Initialize LLM
llm = ChatMistralAI(
    model=CodeAssistantConfig.MODEL_NAME,
    temperature=CodeAssistantConfig.TEMPERATURE
)

# Create chain with structured output
code_gen_chain = llm.with_structured_output(CodeSolution)

# Define the state passed between LangGraph nodes
class GraphState(TypedDict):
    """State passed between LangGraph nodes."""
    error: str
    messages: Annotated[list[AnyMessage], add_messages]
    generation: CodeSolution
    iterations: int
    question: str
    rag_context: str

print("✓ LLM and GraphState initialized")

## 5. Agent Node Functions

In [ ]:
def retrieve_context(state: GraphState) -> dict:
    """Retrieve project context using RAG if available."""
    print("---RETRIEVING CONTEXT---")
    if not rag_retriever:
        return {"rag_context": ""}
    
    question = state["messages"][-1].content
    docs = rag_retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    return {"rag_context": context}

def generate_code(state: GraphState) -> dict:
    """Generate code using the LLM."""
    print("---GENERATING CODE---")
    question = state["messages"][-1].content
    rag_context = state.get("rag_context", "")
    error = state.get("error", "")
    
    # Build prompt with context and error information
    if rag_context:
        system_msg = "You are an expert Python code generator.\n\nProject Context:\n{rag_context}\n\nPrevious Error: {error}\n\nGenerate correct, working Python code."
    else:
        system_msg = "You are an expert Python code generator.\n\nPrevious Error: {error}\n\nGenerate correct, working Python code."
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("user", "{question}")
    ])
    
    response = (prompt | code_gen_chain).invoke({
        "question": question,
        "rag_context": rag_context,
        "error": error
    })
    
    return {
        "generation": response,
        "iterations": state.get("iterations", 0) + 1
    }

def validate_code(state: GraphState) -> dict:
    """Validate generated code in an isolated subprocess."""
    print("---VALIDATING CODE---")
    gen = state["generation"]
    full_code = f"{gen.imports}\n{gen.code}"
    
    try:
        # Execute code in subprocess with timeout
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(full_code)
            temp_file = f.name
        
        try:
            result = subprocess.run(
                ["python", temp_file],
                timeout=CodeAssistantConfig.VALIDATION_TIMEOUT,
                capture_output=True,
                text=True
            )
            
            if result.returncode == 0:
                print("✓ Code validation successful")
                return {"error": "none"}
            else:
                error_msg = result.stderr or result.stdout
                print(f"✗ Code execution failed: {error_msg}")
                return {"error": error_msg}
        finally:
            import os
            os.unlink(temp_file)
            
    except subprocess.TimeoutExpired:
        error_msg = f"Code execution timed out (>{CodeAssistantConfig.VALIDATION_TIMEOUT}s)"
        print(f"✗ {error_msg}")
        return {"error": error_msg}
    except Exception as e:
        print(f"✗ Validation error: {str(e)}")
        return {"error": str(e)}

print("✓ Node functions defined")

## 6. Graph Construction with Error Recovery

In [ ]:
from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
import uuid

def should_continue(state: GraphState) -> Literal["validate_code", "generate_code", END]:
    """Decide whether to continue generation or end."""
    error = state.get("error", "")
    iterations = state.get("iterations", 0)
    
    if error == "none":
        print("→ Code is valid. Finishing...")
        return END
    elif iterations >= CodeAssistantConfig.MAX_ITERATIONS:
        print(f"→ Max iterations ({CodeAssistantConfig.MAX_ITERATIONS}) reached. Finishing...")
        return END
    else:
        print(f"→ Retrying (iteration {iterations + 1}/{CodeAssistantConfig.MAX_ITERATIONS})...")
        return "generate_code"

# Build the LangGraph workflow
builder = StateGraph(GraphState)

# Add nodes
builder.add_node("retrieve_context", retrieve_context)
builder.add_node("generate_code", generate_code)
builder.add_node("validate_code", validate_code)

# Define edges
builder.add_edge(START, "retrieve_context")
builder.add_edge("retrieve_context", "generate_code")
builder.add_edge("generate_code", "validate_code")

# Add conditional edge for retry logic
builder.add_conditional_edges(
    "validate_code",
    should_continue,
    {
        "validate_code": END,  # Successfully validated
        "generate_code": "generate_code",  # Retry generation
        END: END
    }
)

# Compile graph with in-memory checkpointer
graph = builder.compile(checkpointer=InMemorySaver())

print("✓ LangGraph workflow constructed")
print(f"  Max retries: {CodeAssistantConfig.MAX_ITERATIONS}")
print(f"  Timeout per execution: {CodeAssistantConfig.VALIDATION_TIMEOUT}s")

## 7. Example Execution

In [ ]:
# Run the agent with a sample prompt
user_question = "Write a Python function that checks if a number is prime and returns True or False."

print("\n" + "="*70)
print("EXECUTING CODE ASSISTANT AGENT")
print("="*70)
print(f"\nUser Question: {user_question}\n")

# Configure the execution session
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Initialize state
input_state = {
    "messages": [("user", user_question)],
    "iterations": 0,
    "error": "",
    "rag_context": "",
    "question": user_question
}

# Stream the graph execution
final_result = None
for event in graph.stream(input_state, config, stream_mode="values"):
    if "generation" in event and event["generation"]:
        final_result = event["generation"]

print("\n" + "="*70)
if final_result:
    print("GENERATED CODE SOLUTION")
    print("="*70)
    print("\n[Approach]")
    print(final_result.prefix)
    print("\n[Imports]")
    print(final_result.imports)
    print("\n[Code]")
    print(final_result.code)
else:
    print("No solution generated. Check configuration and API keys.")
print("="*70)

## 8. Workflow Visualization

In [ ]:
from IPython.display import Image, display

# Generate and display the graph structure
try:
    print("Generating workflow diagram...\n")
    # 'xray=True' shows internal details of nested nodes
    mermaid_diagram = graph.get_graph(xray=True).draw_mermaid_png()
    display(Image(mermaid_diagram))
    print("\n✓ Workflow diagram generated successfully!")
    print("\nWorkflow stages:")
    print("  1. retrieve_context → Fetch project context via RAG (optional)")
    print("  2. generate_code → Generate code with LLM")
    print("  3. validate_code → Test code in subprocess")
    print("  4. should_continue → Decide: retry or finish")
except Exception as e:
    print(f"⚠ Could not render diagram: {e}")
    print("\nWorkflow stages:")
    print("  1. retrieve_context → Fetch project context via RAG (optional)")
    print("  2. generate_code → Generate code with LLM")
    print("  3. validate_code → Test code in subprocess")
    print("  4. should_continue → Decide: retry or finish")

## 9. CLI Usage

### Run from Command Line

```bash
# Basic usage
python main.py "Write a Python function that returns the Fibonacci sequence"

# With project RAG context
python main.py --rag "Add a function that matches this codebase style"

# With custom provider and model
python main.py --provider openai "Your question here"

# With custom settings
python main.py --model gpt-4 --max-iterations 5 "Your question"
```

### Start Web Application

```bash
# Launch the backend with frontend
python web_main.py

# Then open: http://localhost:8000
```

### Web API Endpoints

- **GET `/`** - Serve the web UI
- **GET `/api/health`** - Health check
- **GET `/api/config`** - Get configuration
- **POST `/api/generate`** - Generate code (requires API key)
- **GET `/api/docs`** - Interactive API documentation (Swagger)

### Example API Request

```bash
curl -X POST http://localhost:8000/api/generate \
  -H "Authorization: Bearer YOUR_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
    "prompt": "Write a function to parse JSON",
    "provider": "mistral",
    "max_iterations": 3
  }'
```

## 10. Project Architecture Summary

### Key Components

#### Core System (LangGraph)
- **GraphState**: Maintains execution state across agent steps
- **retrieve_context**: Optional RAG retrieval for project context
- **generate_code**: LLM-based code generation with structured output
- **validate_code**: Subprocess-based code validation with timeout protection
- **Conditional Logic**: Automatic retry with error feedback

#### LLM Providers Supported
- **Mistral** - Fast and cost-effective
- **OpenAI** - High quality code generation
- **Local Models** - Privacy-first via HuggingFace
- **OpenRouter** - Access to multiple models

#### RAG System (Optional)
- **Vector Store**: Qdrant for scalable similarity search
- **Embeddings**: Mistral embeddings for dense representations
- **Retrieval**: Context-aware code generation
- **Indexing**: `scripts/index_project_rag.py` for project analysis

#### API Layer (FastAPI)
- Authentication with bearer tokens
- Rate limiting and request caps
- CORS support
- Benchmarking and metrics tracking
- Error classification for debugging

### Features
✓ Code correction with up to 3 retry iterations
✓ Timeout protection (configurable per execution)
✓ Structured output validation
✓ Project-aware context retrieval
✓ Benchmark reporting
✓ User API key management
✓ Sandboxed code execution
✓ Failure diagnostics and logging

### Performance Metrics
The system generates benchmarks for:
- **Latency**: Time from prompt to solution
- **Accuracy**: Code execution success rate
- **Iterations**: Average retry count per request
- **Provider Comparison**: Mistral vs OpenAI performance

Benchmark reports are saved to `artifacts/benchmark_reports/`

### Next Steps
1. **Local Development**: `python main.py "your prompt"`
2. **Web UI**: `python web_main.py` → http://localhost:8000
3. **Project Context**: `python scripts/index_project_rag.py`
4. **Fine-tuning**: `python scripts/train_local_lora.py`
5. **Deployment**: See `docs/DEPLOYMENT.md` for hosting on Render/Docker

### Documentation
- `docs/ARCHITECTURE.md` - System design details
- `docs/DEPLOYMENT.md` - Production deployment guide
- `README.md` - Quick start and features overview

## 11. PDF Ingestion Integration (OpenDataLoader PDF)

This project now includes first-party PDF ingestion for RAG.

- Input PDFs: `data/pdfs/`
- Converted output: `docs/pdf_ingested/`
- Script: `scripts/ingest_project_pdfs.py`

### Example Commands

```powershell
python scripts/ingest_project_pdfs.py --reindex-rag
```

Custom paths:

```powershell
python scripts/ingest_project_pdfs.py --input data/pdfs --input archive/customer_docs --output-dir docs/pdf_ingested --format markdown,json --reindex-rag
```

Expected output shape:

```text
PDF ingestion complete: <N> PDF(s) converted from <M> input path(s) into <output_dir>.
RAG index refreshed: <files> file(s), <chunks> chunk(s) at <qdrant_path>.
```


In [ ]:
# Quick demo: show ingestion CLI options (safe to run)
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "scripts/ingest_project_pdfs.py", "--help"],
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
